### Silver Layer - Data Cleaning & Feature Engineering

## Objective

The Silver layer transforms the raw Bronze datasets into a clean, standardized, and analysis-ready dataset.

This stage performs data cleaning, validation, preprocessing, and feature engineering. It also generates row-level KPIs that will be used later in the Gold Layer for Data Warehouse modeling, dashboards, and machine learning.

---

## Processing Steps

The Silver layer performs the following tasks:

### 1. Load Bronze Data
- Load the four raw datasets generated in the Bronze layer.
- Add a timeframe identifier.
- Merge all datasets into one unified dataset.

### 2. Data Cleaning
- Inspect the dataset.
- Convert timestamps into datetime format.
- Extract temporal attributes.
- Remove unnecessary columns.
- Handle missing values.
- Remove duplicate records.
- Validate numerical columns.

### 3. Feature Engineering
Generate analytical features including:

- Price KPIs
- Volume KPIs
- Trend Indicators
- Support & Resistance
- Breakout Signals
- Market Behavior Signals

### 4. Final Validation
- Remove rows affected by rolling calculations.
- Keep only analytical columns.
- Save the cleaned dataset.

---

## Output

**Input**

- bronze_15m.csv
- bronze_1h.csv
- bronze_4h.csv
- bronze_1d.csv

**Output**

- silver_bitcoin.csv

In [4]:
#import required libraries
import pandas as pd
import numpy as np


In [5]:
# Load Bronze datasets
df_15m = pd.read_csv(r"C:\Users\admin\Desktop\bronze layer\bronze_15m.csv")
df_1h  = pd.read_csv(r"C:\Users\admin\Desktop\bronze layer\bronze_1h.csv")
df_4h  = pd.read_csv(r"C:\Users\admin\Desktop\bronze layer\bronze_4h.csv")
df_1d  = pd.read_csv(r"C:\Users\admin\Desktop\bronze layer\bronze_1d.csv")


### Data Preprocessing


In [6]:
# Add timeframe labels
df_15m["timeframe"] = "15m"
df_1h["timeframe"] = "1h"
df_4h["timeframe"] = "4h"
df_1d["timeframe"] = "1d"

# Merge datasets
df = pd.concat(
    [df_15m, df_1h, df_4h, df_1d],
    ignore_index=True
)

In [ ]:
df.head()

,Open time,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore,timeframe
0,2018-01-01 00:00:00.000000,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000,1.675545e+06,1572,63.227133,8.576108e+05,0,15m
1,2018-01-01 00:15:00.000000,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000,1.321757e+06,1461,47.686389,6.422812e+05,0,15m
2,2018-01-01 00:30:00.000000,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000,1.078825e+06,1000,43.710406,5.900347e+05,0,15m
3,2018-01-01 00:45:00.000000,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000,1.917783e+06,1195,73.897993,1.000614e+06,0,15m
4,2018-01-01 01:00:00.000000,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000,9.778198e+05,898,34.257652,4.618369e+05,0,15m


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391510 entries, 0 to 391509
Data columns (total 13 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Open time                     391507 non-null  object 
 1   Open                          391510 non-null  float64
 2   High                          391510 non-null  float64
 3   Low                           391510 non-null  float64
 4   Close                         391510 non-null  float64
 5   Volume                        391510 non-null  float64
 6   Close time                    391507 non-null  object 
 7   Quote asset volume            391510 non-null  float64
 8   Number of trades              391510 non-null  int64  
 9   Taker buy base asset volume   391510 non-null  float64
 10  Taker buy quote asset volume  391510 non-null  float64
 11  Ignore                        391510 non-null  int64  
 12  timeframe                     391510 non-nul

In [ ]:
df.describe()

,Open,High,Low,Close,Volume,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore
count,391510.000000,391510.000000,391510.000000,391510.000000,391510.000000,3.915100e+05,3.915100e+05,391510.000000,3.915100e+05,391510.0
mean,39325.245093,39432.640840,39213.157351,39325.886893,1969.506244,5.652247e+07,6.305209e+04,978.020394,2.791147e+07,0.0
std,32556.019527,32622.384136,32487.781864,32556.183340,9630.001303,2.517230e+08,2.731358e+05,4790.287864,1.248749e+08,0.0
min,3166.110000,3174.780000,3156.260000,3167.070000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.0
25%,9696.070000,9727.970000,9661.160000,9695.917500,220.137512,4.994165e+06,6.176000e+03,108.214681,2.384261e+06,0.0
50%,29320.140000,29373.595000,29268.725000,29320.400000,469.282420,1.402867e+07,1.555650e+04,232.889784,6.812442e+06,0.0
75%,62203.665000,62397.677500,62027.692500,62204.190000,1261.887156,3.800075e+07,4.372500e+04,629.808423,1.874148e+07,0.0
max,126011.180000,126199.630000,125648.010000,126011.180000,760705.362783,1.746531e+10,1.522359e+07,374775.574085,8.783916e+09,0.0


In [10]:
df.shape

(391510, 13)

In [ ]:
# Convert timestamps to datetime
df["Open time"] = pd.to_datetime(df["Open time"], format="mixed", utc=True)
df["Close time"] = pd.to_datetime(df["Close time"], format="mixed", utc=True)

# Sort records
df = df.sort_values("Open time").reset_index(drop=True)

In [ ]:
df.head(50)

,Open time,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore,timeframe
0,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000+00:00,1.675545e+06,1572,63.227133,8.576108e+05,0,15m
1,2018-01-01 00:00:00+00:00,13715.65,13715.65,13155.38,13410.03,1676.204807,2018-01-01 03:59:59.999000+00:00,2.251607e+07,19438,739.518666,9.937537e+06,0,4h
2,2018-01-01 00:00:00+00:00,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000+00:00,1.147997e+08,105595,3961.938946,5.280975e+07,0,1d
3,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999000+00:00,5.993910e+06,5228,228.521921,3.090541e+06,0,1h
4,2018-01-01 00:15:00+00:00,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000+00:00,1.321757e+06,1461,47.686389,6.422812e+05,0,15m
5,2018-01-01 00:30:00+00:00,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000+00:00,1.078825e+06,1000,43.710406,5.900347e+05,0,15m
6,2018-01-01 00:45:00+00:00,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000+00:00,1.917783e+06,1195,73.897993,1.000614e+06,0,15m
7,2018-01-01 01:00:00+00:00,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000+00:00,9.778198e+05,898,34.257652,4.618369e+05,0,15m
8,2018-01-01 01:00:00+00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999000+00:00,5.154522e+06,4534,180.840403,2.430449e+06,0,1h
9,2018-01-01 01:15:00+00:00,13469.99,13595.89,13445.63,13560.00,87.861758,2018-01-01 01:29:59.999000+00:00,1.189941e+06,939,45.135957,6.111437e+05,0,15m


In [ ]:
# Remove unnecessary column
df = df.drop(columns=["Ignore"])

In [ ]:
# Extract calendar attributes
df["year"] = df["Open time"].dt.year
df["month"] = df["Open time"].dt.month
df["day"] = df["Open time"].dt.day
df["quarter"] = df["Open time"].dt.quarter

In [ ]:
# Check missing values
df[["year","month","day","quarter"]].isna().sum()

year       3
month      3
day        3
quarter    3
dtype: int64

In [ ]:
# display the first five rows that contain at least one missing value (NaN) in any column
df[df.isna().any(axis=1)].head()

,Open time,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,timeframe,year,month,day,quarter
391507,NaT,66226.07,66269.52,66116.59,66269.52,100.41320,NaT,6.646592e+06,25639,66.32040,4.389696e+06,15m,NaN,NaN,NaN,NaN
391508,NaT,66368.15,66476.64,66076.56,66206.34,520.15584,NaT,3.446694e+07,160615,258.69663,1.714831e+07,1h,NaN,NaN,NaN,NaN
391509,NaT,67292.15,67292.15,66348.00,66614.35,3808.29082,NaT,2.547718e+08,642459,1581.91921,1.058234e+08,4h,NaN,NaN,NaN,NaN


In [ ]:
# Remove invalid timestamp rows
df = df.dropna(subset=["Open time"])
# Reset index
df = df.reset_index(drop=True)

In [ ]:
# Convert calendar fields to integer
df["year"] = df["Open time"].dt.year.astype(int)
df["month"] = df["Open time"].dt.month.astype(int)
df["day"] = df["Open time"].dt.day.astype(int)
df["quarter"] = df["Open time"].dt.quarter.astype(int)

In [ ]:
df.head(50)

,Open time,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,timeframe,year,month,day,quarter
0,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000+00:00,1.675545e+06,1572,63.227133,8.576108e+05,15m,2018,1,1,1
1,2018-01-01 00:00:00+00:00,13715.65,13715.65,13155.38,13410.03,1676.204807,2018-01-01 03:59:59.999000+00:00,2.251607e+07,19438,739.518666,9.937537e+06,4h,2018,1,1,1
2,2018-01-01 00:00:00+00:00,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000+00:00,1.147997e+08,105595,3961.938946,5.280975e+07,1d,2018,1,1,1
3,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999000+00:00,5.993910e+06,5228,228.521921,3.090541e+06,1h,2018,1,1,1
4,2018-01-01 00:15:00+00:00,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000+00:00,1.321757e+06,1461,47.686389,6.422812e+05,15m,2018,1,1,1
5,2018-01-01 00:30:00+00:00,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000+00:00,1.078825e+06,1000,43.710406,5.900347e+05,15m,2018,1,1,1
6,2018-01-01 00:45:00+00:00,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000+00:00,1.917783e+06,1195,73.897993,1.000614e+06,15m,2018,1,1,1
7,2018-01-01 01:00:00+00:00,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000+00:00,9.778198e+05,898,34.257652,4.618369e+05,15m,2018,1,1,1
8,2018-01-01 01:00:00+00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999000+00:00,5.154522e+06,4534,180.840403,2.430449e+06,1h,2018,1,1,1
9,2018-01-01 01:15:00+00:00,13469.99,13595.89,13445.63,13560.00,87.861758,2018-01-01 01:29:59.999000+00:00,1.189941e+06,939,45.135957,6.111437e+05,15m,2018,1,1,1


In [ ]:
df.isna().sum()

Open time                       0
Open                            0
High                            0
Low                             0
Close                           0
Volume                          0
Close time                      0
Quote asset volume              0
Number of trades                0
Taker buy base asset volume     0
Taker buy quote asset volume    0
timeframe                       0
year                            0
month                           0
day                             0
quarter                         0
dtype: int64

In [ ]:
# Check duplicates
df.duplicated().sum()

np.int64(0)

In [ ]:
df.duplicated(subset=["Open time","timeframe"]).sum()

np.int64(6)

In [ ]:
df[df.duplicated(subset=["Open time","timeframe"], keep=False)]

,Open time,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,timeframe,year,month,day,quarter
23343,2018-07-06 00:00:00+00:00,108700.00,109111.00,108518.83,108888.00,1086.391210,2018-07-06 03:59:59.999000+00:00,1.182862e+08,214025,508.738310,5.538955e+07,4h,2018,7,6,3
23344,2018-07-06 00:00:00+00:00,6529.20,6553.00,6511.00,6530.00,4120.321071,2018-07-06 03:59:59.999000+00:00,2.690673e+07,24305,2337.061505,1.526290e+07,4h,2018,7,6,3
23363,2018-07-06 04:00:00+00:00,6530.00,6533.63,6425.00,6478.85,5824.504486,2018-07-06 07:59:59.999000+00:00,3.778332e+07,37316,3043.078755,1.974576e+07,4h,2018,7,6,3
23364,2018-07-06 04:00:00+00:00,108888.00,108978.64,107505.00,107573.29,3616.089460,2018-07-06 07:59:59.999000+00:00,3.905847e+08,627971,1611.098070,1.740189e+08,4h,2018,7,6,3
23470,2018-07-07 00:00:00+00:00,6609.79,6613.99,6581.05,6596.93,766.671591,2018-07-07 00:59:59.999000+00:00,5.055038e+06,6145,417.287181,2.751863e+06,1h,2018,7,7,3
23472,2018-07-07 00:00:00+00:00,107469.22,107813.04,107440.63,107692.01,228.631950,2018-07-07 00:59:59.999000+00:00,2.461733e+07,42593,138.742240,1.493892e+07,1h,2018,7,7,3
23478,2018-07-07 01:00:00+00:00,107692.01,107941.99,107689.91,107911.20,255.841360,2018-07-07 01:59:59.999000+00:00,2.757870e+07,33521,129.597780,1.397040e+07,1h,2018,7,7,3
23479,2018-07-07 01:00:00+00:00,6596.93,6600.00,6561.01,6566.01,887.436249,2018-07-07 01:59:59.999000+00:00,5.837711e+06,6193,488.621779,3.214876e+06,1h,2018,7,7,3
23504,2018-07-07 06:00:00+00:00,6598.00,6600.00,6592.88,6592.90,184.610237,2018-07-07 06:14:59.999000+00:00,1.217864e+06,924,73.630565,4.857697e+05,15m,2018,7,7,3
23505,2018-07-07 06:00:00+00:00,108254.95,108254.96,108090.24,108115.55,64.056100,2018-07-07 06:14:59.999000+00:00,6.927307e+06,10111,21.874150,2.365207e+06,15m,2018,7,7,3


In [ ]:
# Keep the record with the highest number of trades for each duplicated
# Open time and timeframe, then sort the dataset df = df.sort_values("Number of trades", ascending=False)
df = df.drop_duplicates(subset=["Open time","timeframe"], keep="first")
df = df.sort_values("Open time")
df = df.reset_index(drop=True)

In [ ]:
df.dtypes

Open time                       datetime64[ns, UTC]
Open                                        float64
High                                        float64
Low                                         float64
Close                                       float64
Volume                                      float64
Close time                      datetime64[ns, UTC]
Quote asset volume                          float64
Number of trades                              int64
Taker buy base asset volume                 float64
Taker buy quote asset volume                float64
timeframe                                    object
year                                          int64
month                                         int64
day                                           int64
quarter                                       int64
dtype: object

In [ ]:
# Validate numeric columns
(df[["Open","High","Low","Close","Volume"]] < 0).sum()

Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

### Feature Engnineering

#### price

In [27]:
#price movement

df['price_change'] = df['Close'] - df['Open']


# Return
df['return_pct'] = df['price_change'] / df['Open']

# Volatility
df['volatility'] = df['High'] - df['Low']

In [ ]:
df.head(50)

,Open time,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,timeframe,year,month,day,quarter,price_change,return_pct,volatility
0,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000+00:00,1.675545e+06,1572,63.227133,8.576108e+05,15m,2018,1,1,1,-159.50,-0.011629,315.64
1,2018-01-01 00:00:00+00:00,13715.65,13715.65,13155.38,13410.03,1676.204807,2018-01-01 03:59:59.999000+00:00,2.251607e+07,19438,739.518666,9.937537e+06,4h,2018,1,1,1,-305.62,-0.022283,560.27
2,2018-01-01 00:00:00+00:00,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000+00:00,1.147997e+08,105595,3961.938946,5.280975e+07,1d,2018,1,1,1,-335.65,-0.024472,1068.55
3,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999000+00:00,5.993910e+06,5228,228.521921,3.090541e+06,1h,2018,1,1,1,-186.64,-0.013608,315.64
4,2018-01-01 00:15:00+00:00,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000+00:00,1.321757e+06,1461,47.686389,6.422812e+05,15m,2018,1,1,1,-12.63,-0.000933,148.87
5,2018-01-01 00:30:00+00:00,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000+00:00,1.078825e+06,1000,43.710406,5.900347e+05,15m,2018,1,1,1,-29.59,-0.002192,95.37
6,2018-01-01 00:45:00+00:00,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000+00:00,1.917783e+06,1195,73.897993,1.000614e+06,15m,2018,1,1,1,34.36,0.002546,240.87
7,2018-01-01 01:00:00+00:00,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000+00:00,9.778198e+05,898,34.257652,4.618369e+05,15m,2018,1,1,1,-83.36,-0.006162,169.46
8,2018-01-01 01:00:00+00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999000+00:00,5.154522e+06,4534,180.840403,2.430449e+06,1h,2018,1,1,1,-325.93,-0.024091,440.51
9,2018-01-01 01:15:00+00:00,13469.99,13595.89,13445.63,13560.00,87.861758,2018-01-01 01:29:59.999000+00:00,1.189941e+06,939,45.135957,6.111437e+05,15m,2018,1,1,1,90.01,0.006682,150.26


#### trend


In [29]:
# Trend Direction
df['trend_direction'] = np.where(df['price_change'] > 0, 1, -1)

# Trend Streak
df['trend_streak'] = df.groupby('timeframe')['trend_direction'].transform(
    lambda x: (x != x.shift()).cumsum()
)

# Trend Length
df['trend_length'] = df.groupby(['timeframe', 'trend_streak']).cumcount() + 1

# Trend Strength
df['trend_strength'] = np.where(
    df['Close'] != 0,
    (df['price_change'] / df['Close']) * df['Volume'],
    0
)

In [30]:
df.head(10)


,Open time,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,...,month,day,quarter,price_change,return_pct,volatility,trend_direction,trend_streak,trend_length,trend_strength
0,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000+00:00,1.675545e+06,1572,63.227133,...,1,1,1,-159.50,-0.011629,315.64,-1,1,1,-1.454451
1,2018-01-01 00:00:00+00:00,13715.65,13715.65,13155.38,13410.03,1676.204807,2018-01-01 03:59:59.999000+00:00,2.251607e+07,19438,739.518666,...,1,1,1,-305.62,-0.022283,560.27,-1,1,1,-38.201385
2,2018-01-01 00:00:00+00:00,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000+00:00,1.147997e+08,105595,3961.938946,...,1,1,1,-335.65,-0.024472,1068.55,-1,1,1,-215.987911
3,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999000+00:00,5.993910e+06,5228,228.521921,...,1,1,1,-186.64,-0.013608,315.64,-1,1,1,-6.116338
4,2018-01-01 00:15:00+00:00,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000+00:00,1.321757e+06,1461,47.686389,...,1,1,1,-12.63,-0.000933,148.87,-1,1,2,-0.091669
5,2018-01-01 00:30:00+00:00,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000+00:00,1.078825e+06,1000,43.710406,...,1,1,1,-29.59,-0.002192,95.37,-1,1,3,-0.175523
6,2018-01-01 00:45:00+00:00,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000+00:00,1.917783e+06,1195,73.897993,...,1,1,1,34.36,0.002546,240.87,1,2,1,0.359879
7,2018-01-01 01:00:00+00:00,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000+00:00,9.778198e+05,898,34.257652,...,1,1,1,-83.36,-0.006162,169.46,-1,3,1,-0.449717
8,2018-01-01 01:00:00+00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999000+00:00,5.154522e+06,4534,180.840403,...,1,1,1,-325.93,-0.024091,440.51,-1,1,2,-9.471923
9,2018-01-01 01:15:00+00:00,13469.99,13595.89,13445.63,13560.00,87.861758,2018-01-01 01:29:59.999000+00:00,1.189941e+06,939,45.135957,...,1,1,1,90.01,0.006682,150.26,1,4,1,0.583218


#### volume & market activity


In [31]:
# Buy Volume
df = df.rename(columns={'Taker buy base asset volume': 'buy_volume'})

# Sell Volume
df['sell_volume'] = df['Volume'] - df['buy_volume']


# Buy Ratio
df['buy_ratio'] = np.where(
    df['Volume'] != 0,
    df['buy_volume'] / df['Volume'],
    0
)

# Sell Ratio
df['sell_ratio'] = np.where(
    df['Volume'] != 0,
    df['sell_volume'] / df['Volume'],
    0
)

# Net Order Flow
df['order_flow'] = df['buy_volume'] - df['sell_volume']

# Capital Flow
df.rename(columns={'Quote asset volume': 'capital_flow'}, inplace=True)
#trade activity
df.rename(columns={'Number of trades': 'trade_activity'}, inplace=True)

In [32]:
df.head(10)


,Open time,Open,High,Low,Close,Volume,Close time,capital_flow,trade_activity,buy_volume,...,return_pct,volatility,trend_direction,trend_streak,trend_length,trend_strength,sell_volume,buy_ratio,sell_ratio,order_flow
0,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000+00:00,1.675545e+06,1572,63.227133,...,-0.011629,315.64,-1,1,1,-1.454451,60.388880,0.511480,0.488520,2.838253
1,2018-01-01 00:00:00+00:00,13715.65,13715.65,13155.38,13410.03,1676.204807,2018-01-01 03:59:59.999000+00:00,2.251607e+07,19438,739.518666,...,-0.022283,560.27,-1,1,1,-38.201385,936.686141,0.441186,0.558814,-197.167475
2,2018-01-01 00:00:00+00:00,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000+00:00,1.147997e+08,105595,3961.938946,...,-0.024472,1068.55,-1,1,1,-215.987911,4647.976898,0.460160,0.539840,-686.037952
3,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999000+00:00,5.993910e+06,5228,228.521921,...,-0.013608,315.64,-1,1,1,-6.116338,214.834278,0.515436,0.484564,13.687643
4,2018-01-01 00:15:00+00:00,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000+00:00,1.321757e+06,1461,47.686389,...,-0.000933,148.87,-1,1,2,-0.091669,50.450041,0.485919,0.514081,-2.763652
5,2018-01-01 00:30:00+00:00,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000+00:00,1.078825e+06,1000,43.710406,...,-0.002192,95.37,-1,1,3,-0.175523,36.193631,0.547036,0.452964,7.516775
6,2018-01-01 00:45:00+00:00,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000+00:00,1.917783e+06,1195,73.897993,...,0.002546,240.87,1,2,1,0.359879,67.801726,0.521511,0.478489,6.096267
7,2018-01-01 01:00:00+00:00,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000+00:00,9.778198e+05,898,34.257652,...,-0.006162,169.46,-1,3,1,-0.449717,38.279881,0.472275,0.527725,-4.022229
8,2018-01-01 01:00:00+00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999000+00:00,5.154522e+06,4534,180.840403,...,-0.024091,440.51,-1,1,2,-9.471923,202.856603,0.471310,0.528690,-22.016200
9,2018-01-01 01:15:00+00:00,13469.99,13595.89,13445.63,13560.00,87.861758,2018-01-01 01:29:59.999000+00:00,1.189941e+06,939,45.135957,...,0.006682,150.26,1,4,1,0.583218,42.725801,0.513716,0.486284,2.410156


#### support & resistance


In [33]:
df['support'] = df['Close'] - df['Low']
df['resistance'] = df['High'] - df['Close']

In [34]:
df.head(10)


,Open time,Open,High,Low,Close,Volume,Close time,capital_flow,trade_activity,buy_volume,...,trend_direction,trend_streak,trend_length,trend_strength,sell_volume,buy_ratio,sell_ratio,order_flow,support,resistance
0,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000+00:00,1.675545e+06,1572,63.227133,...,-1,1,1,-1.454451,60.388880,0.511480,0.488520,2.838253,156.14,159.50
1,2018-01-01 00:00:00+00:00,13715.65,13715.65,13155.38,13410.03,1676.204807,2018-01-01 03:59:59.999000+00:00,2.251607e+07,19438,739.518666,...,-1,1,1,-38.201385,936.686141,0.441186,0.558814,-197.167475,254.65,305.62
2,2018-01-01 00:00:00+00:00,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000+00:00,1.147997e+08,105595,3961.938946,...,-1,1,1,-215.987911,4647.976898,0.460160,0.539840,-686.037952,630.00,438.55
3,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999000+00:00,5.993910e+06,5228,228.521921,...,-1,1,1,-6.116338,214.834278,0.515436,0.484564,13.687643,129.00,186.64
4,2018-01-01 00:15:00+00:00,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000+00:00,1.321757e+06,1461,47.686389,...,-1,1,2,-0.091669,50.450041,0.485919,0.514081,-2.763652,119.12,29.75
5,2018-01-01 00:30:00+00:00,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000+00:00,1.078825e+06,1000,43.710406,...,-1,1,3,-0.175523,36.193631,0.547036,0.452964,7.516775,20.41,74.96
6,2018-01-01 00:45:00+00:00,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000+00:00,1.917783e+06,1195,73.897993,...,1,2,1,0.359879,67.801726,0.521511,0.478489,6.096267,79.01,161.86
7,2018-01-01 01:00:00+00:00,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000+00:00,9.778198e+05,898,34.257652,...,-1,3,1,-0.449717,38.279881,0.472275,0.527725,-4.022229,43.35,126.11
8,2018-01-01 01:00:00+00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999000+00:00,5.154522e+06,4534,180.840403,...,-1,1,2,-9.471923,202.856603,0.471310,0.528690,-22.016200,47.68,392.83
9,2018-01-01 01:15:00+00:00,13469.99,13595.89,13445.63,13560.00,87.861758,2018-01-01 01:29:59.999000+00:00,1.189941e+06,939,45.135957,...,1,4,1,0.583218,42.725801,0.513716,0.486284,2.410156,114.37,35.89


In [35]:
df['avg_volume_20'] = df.groupby('timeframe')['Volume'].transform(
    lambda x: x.rolling(20).mean()
)

df['avg_volatility_20'] = df.groupby('timeframe')['volatility'].transform(
    lambda x: x.rolling(20).mean()
)

In [36]:
df.head(10)

,Open time,Open,High,Low,Close,Volume,Close time,capital_flow,trade_activity,buy_volume,...,trend_length,trend_strength,sell_volume,buy_ratio,sell_ratio,order_flow,support,resistance,avg_volume_20,avg_volatility_20
0,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000+00:00,1.675545e+06,1572,63.227133,...,1,-1.454451,60.388880,0.511480,0.488520,2.838253,156.14,159.50,NaN,NaN
1,2018-01-01 00:00:00+00:00,13715.65,13715.65,13155.38,13410.03,1676.204807,2018-01-01 03:59:59.999000+00:00,2.251607e+07,19438,739.518666,...,1,-38.201385,936.686141,0.441186,0.558814,-197.167475,254.65,305.62,NaN,NaN
2,2018-01-01 00:00:00+00:00,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000+00:00,1.147997e+08,105595,3961.938946,...,1,-215.987911,4647.976898,0.460160,0.539840,-686.037952,630.00,438.55,NaN,NaN
3,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999000+00:00,5.993910e+06,5228,228.521921,...,1,-6.116338,214.834278,0.515436,0.484564,13.687643,129.00,186.64,NaN,NaN
4,2018-01-01 00:15:00+00:00,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000+00:00,1.321757e+06,1461,47.686389,...,2,-0.091669,50.450041,0.485919,0.514081,-2.763652,119.12,29.75,NaN,NaN
5,2018-01-01 00:30:00+00:00,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000+00:00,1.078825e+06,1000,43.710406,...,3,-0.175523,36.193631,0.547036,0.452964,7.516775,20.41,74.96,NaN,NaN
6,2018-01-01 00:45:00+00:00,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000+00:00,1.917783e+06,1195,73.897993,...,1,0.359879,67.801726,0.521511,0.478489,6.096267,79.01,161.86,NaN,NaN
7,2018-01-01 01:00:00+00:00,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000+00:00,9.778198e+05,898,34.257652,...,1,-0.449717,38.279881,0.472275,0.527725,-4.022229,43.35,126.11,NaN,NaN
8,2018-01-01 01:00:00+00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999000+00:00,5.154522e+06,4534,180.840403,...,2,-9.471923,202.856603,0.471310,0.528690,-22.016200,47.68,392.83,NaN,NaN
9,2018-01-01 01:15:00+00:00,13469.99,13595.89,13445.63,13560.00,87.861758,2018-01-01 01:29:59.999000+00:00,1.189941e+06,939,45.135957,...,1,0.583218,42.725801,0.513716,0.486284,2.410156,114.37,35.89,NaN,NaN


In [37]:
#breakout up and down

df['rolling_high_20'] = df.groupby('timeframe')['High'].transform(
    lambda x: x.rolling(20).max().shift(1)
)
df['rolling_low_20'] = df.groupby('timeframe')['Low'].transform(
    lambda x: x.rolling(20).min().shift(1)
)

df['breakout_up'] = np.where(
    (df['Close'] > df['rolling_high_20']) &
    (df['Volume'] > df['avg_volume_20']),
    1, 0
)

df['breakout_down'] = np.where(
    (df['Close'] < df['rolling_low_20']) &
    (df['Volume'] > df['avg_volume_20']),
    1, 0
)

In [38]:

#market behavior

#high volatility
df['high_volatility'] = np.where(
    df['volatility'] > df['avg_volatility_20'],
    1, 0
)

#volatility streak
df['volatility_streak'] = df.groupby('timeframe')['high_volatility'].transform(
    lambda x: (x != x.shift()).cumsum()
)
#high volatility length
df['high_volatility_length'] = df.groupby(['timeframe', 'volatility_streak']).cumcount() + 1

#compression
df['compression'] = np.where(
    (df['volatility'] < df['avg_volatility_20'] * 0.5) &
    (df['Volume'] < df['avg_volume_20']),
    1, 0
)

In [39]:
#volume spike
df['volume_spike'] = np.where(
    (df['Volume'] > df['avg_volume_20'] * 1.5) &
    (df['return_pct'].abs() > 0.01),
    1, 0
)

#overheating
df['overheating'] = np.where(
    (df['buy_ratio'] > 0.6) &
    (df['Volume'] > df['avg_volume_20'] * 1.5) &
    (df['return_pct'] > 0),
    1, 0
)

In [40]:
df.head(10)


,Open time,Open,High,Low,Close,Volume,Close time,capital_flow,trade_activity,buy_volume,...,rolling_high_20,rolling_low_20,breakout_up,breakout_down,high_volatility,volatility_streak,high_volatility_length,compression,volume_spike,overheating
0,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000+00:00,1.675545e+06,1572,63.227133,...,NaN,NaN,0,0,0,1,1,0,0,0
1,2018-01-01 00:00:00+00:00,13715.65,13715.65,13155.38,13410.03,1676.204807,2018-01-01 03:59:59.999000+00:00,2.251607e+07,19438,739.518666,...,NaN,NaN,0,0,0,1,1,0,0,0
2,2018-01-01 00:00:00+00:00,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000+00:00,1.147997e+08,105595,3961.938946,...,NaN,NaN,0,0,0,1,1,0,0,0
3,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999000+00:00,5.993910e+06,5228,228.521921,...,NaN,NaN,0,0,0,1,1,0,0,0
4,2018-01-01 00:15:00+00:00,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000+00:00,1.321757e+06,1461,47.686389,...,NaN,NaN,0,0,0,1,2,0,0,0
5,2018-01-01 00:30:00+00:00,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000+00:00,1.078825e+06,1000,43.710406,...,NaN,NaN,0,0,0,1,3,0,0,0
6,2018-01-01 00:45:00+00:00,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000+00:00,1.917783e+06,1195,73.897993,...,NaN,NaN,0,0,0,1,4,0,0,0
7,2018-01-01 01:00:00+00:00,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000+00:00,9.778198e+05,898,34.257652,...,NaN,NaN,0,0,0,1,5,0,0,0
8,2018-01-01 01:00:00+00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999000+00:00,5.154522e+06,4534,180.840403,...,NaN,NaN,0,0,0,1,2,0,0,0
9,2018-01-01 01:15:00+00:00,13469.99,13595.89,13445.63,13560.00,87.861758,2018-01-01 01:29:59.999000+00:00,1.189941e+06,939,45.135957,...,NaN,NaN,0,0,0,1,6,0,0,0


In [ ]:
#whale activity
rows_per_year = {"15m": 35040, "1h": 8760, "4h": 2190, "1d": 365}

def stable_whale_threshold(group):
    tf = group.name
    window = rows_per_year.get(tf, 365)
    return group.rolling(window=window, min_periods=30).quantile(0.95)

df['whale_threshold'] = df.groupby('timeframe')['Volume'].transform(stable_whale_threshold)


df['avg_trade_activity_20'] = df.groupby('timeframe')['trade_activity'].transform(
    lambda x: x.rolling(20).mean()
)


df['whale_activity'] = np.where(
    (df['Volume'] > df['whale_threshold']) &
    (df['trade_activity'] < df['avg_trade_activity_20']),
    1, 0
)

In [42]:
df.head(10)


,Open time,Open,High,Low,Close,Volume,Close time,capital_flow,trade_activity,buy_volume,...,breakout_down,high_volatility,volatility_streak,high_volatility_length,compression,volume_spike,overheating,whale_threshold,avg_trade_activity_20,whale_activity
0,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000+00:00,1.675545e+06,1572,63.227133,...,0,0,1,1,0,0,0,NaN,NaN,0
1,2018-01-01 00:00:00+00:00,13715.65,13715.65,13155.38,13410.03,1676.204807,2018-01-01 03:59:59.999000+00:00,2.251607e+07,19438,739.518666,...,0,0,1,1,0,0,0,NaN,NaN,0
2,2018-01-01 00:00:00+00:00,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000+00:00,1.147997e+08,105595,3961.938946,...,0,0,1,1,0,0,0,NaN,NaN,0
3,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999000+00:00,5.993910e+06,5228,228.521921,...,0,0,1,1,0,0,0,NaN,NaN,0
4,2018-01-01 00:15:00+00:00,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000+00:00,1.321757e+06,1461,47.686389,...,0,0,1,2,0,0,0,NaN,NaN,0
5,2018-01-01 00:30:00+00:00,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000+00:00,1.078825e+06,1000,43.710406,...,0,0,1,3,0,0,0,NaN,NaN,0
6,2018-01-01 00:45:00+00:00,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000+00:00,1.917783e+06,1195,73.897993,...,0,0,1,4,0,0,0,NaN,NaN,0
7,2018-01-01 01:00:00+00:00,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000+00:00,9.778198e+05,898,34.257652,...,0,0,1,5,0,0,0,NaN,NaN,0
8,2018-01-01 01:00:00+00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999000+00:00,5.154522e+06,4534,180.840403,...,0,0,1,2,0,0,0,NaN,NaN,0
9,2018-01-01 01:15:00+00:00,13469.99,13595.89,13445.63,13560.00,87.861758,2018-01-01 01:29:59.999000+00:00,1.189941e+06,939,45.135957,...,0,0,1,6,0,0,0,NaN,NaN,0


In [ ]:
#check&load
df.isna().sum().sort_values(ascending=False)

whale_threshold                 116
rolling_high_20                  80
rolling_low_20                   80
avg_volatility_20                76
avg_trade_activity_20            76
avg_volume_20                    76
Close                             0
Volume                            0
Open time                         0
Open                              0
High                              0
Low                               0
timeframe                         0
Taker buy quote asset volume      0
buy_volume                        0
trade_activity                    0
capital_flow                      0
Close time                        0
year                              0
month                             0
trend_streak                      0
trend_length                      0
day                               0
quarter                           0
price_change                      0
return_pct                        0
volatility                        0
trend_direction             

In [ ]:
df[df.isna().any(axis=1)].head(50)

,Open time,Open,High,Low,Close,Volume,Close time,capital_flow,trade_activity,buy_volume,...,breakout_down,high_volatility,volatility_streak,high_volatility_length,compression,volume_spike,overheating,whale_threshold,avg_trade_activity_20,whale_activity
0,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13556.15,123.616013,2018-01-01 00:14:59.999000+00:00,1.675545e+06,1572,63.227133,...,0,0,1,1,0,0,0,NaN,NaN,0
1,2018-01-01 00:00:00+00:00,13715.65,13715.65,13155.38,13410.03,1676.204807,2018-01-01 03:59:59.999000+00:00,2.251607e+07,19438,739.518666,...,0,0,1,1,0,0,0,NaN,NaN,0
2,2018-01-01 00:00:00+00:00,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000+00:00,1.147997e+08,105595,3961.938946,...,0,0,1,1,0,0,0,NaN,NaN,0
3,2018-01-01 00:00:00+00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999000+00:00,5.993910e+06,5228,228.521921,...,0,0,1,1,0,0,0,NaN,NaN,0
4,2018-01-01 00:15:00+00:00,13533.75,13550.87,13402.00,13521.12,98.136430,2018-01-01 00:29:59.999000+00:00,1.321757e+06,1461,47.686389,...,0,0,1,2,0,0,0,NaN,NaN,0
5,2018-01-01 00:30:00+00:00,13500.00,13545.37,13450.00,13470.41,79.904037,2018-01-01 00:44:59.999000+00:00,1.078825e+06,1000,43.710406,...,0,0,1,3,0,0,0,NaN,NaN,0
6,2018-01-01 00:45:00+00:00,13494.65,13690.87,13450.00,13529.01,141.699719,2018-01-01 00:59:59.999000+00:00,1.917783e+06,1195,73.897993,...,0,0,1,4,0,0,0,NaN,NaN,0
7,2018-01-01 01:00:00+00:00,13528.99,13571.74,13402.28,13445.63,72.537533,2018-01-01 01:14:59.999000+00:00,9.778198e+05,898,34.257652,...,0,0,1,5,0,0,0,NaN,NaN,0
8,2018-01-01 01:00:00+00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999000+00:00,5.154522e+06,4534,180.840403,...,0,0,1,2,0,0,0,NaN,NaN,0
9,2018-01-01 01:15:00+00:00,13469.99,13595.89,13445.63,13560.00,87.861758,2018-01-01 01:29:59.999000+00:00,1.189941e+06,939,45.135957,...,0,0,1,6,0,0,0,NaN,NaN,0


In [45]:

rolling_cols = ['avg_volume_20', 'avg_volatility_20', 'rolling_high_20', 'rolling_low_20', 'avg_trade_activity_20', 'whale_threshold']
df = df.dropna(subset=rolling_cols).reset_index(drop=True)

In [ ]:
df.head(10)

,Open time,Open,High,Low,Close,Volume,Close time,capital_flow,trade_activity,buy_volume,...,breakout_down,high_volatility,volatility_streak,high_volatility_length,compression,volume_spike,overheating,whale_threshold,avg_trade_activity_20,whale_activity
0,2018-01-01 07:15:00+00:00,13774.99,13775.99,13600.00,13698.00,74.453320,2018-01-01 07:29:59.999000+00:00,1.019043e+06,1312,41.686674,...,0,1,6,2,0,0,0,145.675994,1171.05,0
1,2018-01-01 07:30:00+00:00,13698.00,13775.00,13590.00,13644.95,89.776654,2018-01-01 07:44:59.999000+00:00,1.228408e+06,895,41.665985,...,0,1,6,3,0,0,0,145.314514,1162.10,0
2,2018-01-01 07:45:00+00:00,13644.97,13659.97,13555.02,13570.35,43.920484,2018-01-01 07:59:59.999000+00:00,5.972011e+05,814,21.261855,...,0,0,7,1,0,0,0,144.953035,1142.50,0
3,2018-01-01 08:00:00+00:00,13569.98,13665.00,13520.00,13656.23,58.542067,2018-01-01 08:14:59.999000+00:00,7.948098e+05,919,20.562018,...,0,1,8,1,0,0,0,144.591555,1134.95,0
4,2018-01-01 08:15:00+00:00,13656.23,13735.24,13610.27,13632.89,58.900513,2018-01-01 08:29:59.999000+00:00,8.054309e+05,869,34.862743,...,0,0,9,1,0,0,0,144.230076,1107.65,0
5,2018-01-01 08:30:00+00:00,13614.33,13649.96,13400.00,13434.71,104.766871,2018-01-01 08:44:59.999000+00:00,1.413295e+06,1204,46.829395,...,0,1,10,1,0,0,0,143.868596,1115.30,0
6,2018-01-01 08:45:00+00:00,13415.00,13508.69,13414.76,13499.99,49.604102,2018-01-01 08:59:59.999000+00:00,6.684083e+05,741,20.555540,...,0,0,11,1,0,0,0,143.507117,1089.70,0
7,2018-01-01 09:00:00+00:00,13499.97,13599.98,13459.11,13589.83,55.827255,2018-01-01 09:14:59.999000+00:00,7.554221e+05,825,24.819559,...,0,1,12,1,0,0,0,143.145637,1068.20,0
8,2018-01-01 09:15:00+00:00,13556.88,13650.00,13531.19,13549.77,55.580387,2018-01-01 09:29:59.999000+00:00,7.552835e+05,826,27.737784,...,0,0,13,1,0,0,0,142.784158,1058.40,0
9,2018-01-01 09:30:00+00:00,13531.82,13610.00,13510.01,13599.99,50.137197,2018-01-01 09:44:59.999000+00:00,6.803677e+05,696,20.799501,...,0,0,13,2,0,0,0,142.422678,1042.90,0


In [ ]:
df.isna().sum().sort_values(ascending=False)

Open time                       0
Open                            0
High                            0
Low                             0
Close                           0
Volume                          0
Close time                      0
capital_flow                    0
trade_activity                  0
buy_volume                      0
Taker buy quote asset volume    0
timeframe                       0
year                            0
month                           0
day                             0
quarter                         0
price_change                    0
return_pct                      0
volatility                      0
trend_direction                 0
trend_streak                    0
trend_length                    0
trend_strength                  0
sell_volume                     0
buy_ratio                       0
sell_ratio                      0
order_flow                      0
support                         0
resistance                      0
avg_volume_20 

In [ ]:
# Review dataset

df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391385 entries, 0 to 391384
Data columns (total 44 columns):
 #   Column                        Non-Null Count   Dtype              
---  ------                        --------------   -----              
 0   Open time                     391385 non-null  datetime64[ns, UTC]
 1   Open                          391385 non-null  float64            
 2   High                          391385 non-null  float64            
 3   Low                           391385 non-null  float64            
 4   Close                         391385 non-null  float64            
 5   Volume                        391385 non-null  float64            
 6   Close time                    391385 non-null  datetime64[ns, UTC]
 7   capital_flow                  391385 non-null  float64            
 8   trade_activity                391385 non-null  int64              
 9   buy_volume                    391385 non-null  float64            
 10  Taker buy quote asse

,Open,High,Low,Close,Volume,capital_flow,trade_activity,buy_volume,Taker buy quote asset volume,year,...,breakout_down,high_volatility,volatility_streak,high_volatility_length,compression,volume_spike,overheating,whale_threshold,avg_trade_activity_20,whale_activity
count,391385.000000,391385.000000,391385.000000,391385.000000,391385.000000,3.913850e+05,3.913850e+05,391385.000000,3.913850e+05,391385.000000,...,391385.000000,391385.000000,391385.000000,391385.000000,391385.000000,391385.000000,391385.000000,391385.000000,3.913850e+05,391385.000000
mean,39333.158370,39440.500293,39221.160604,39333.803455,1967.862692,5.651153e+07,6.305021e+04,977.185761,2.790568e+07,2021.748112,...,0.035045,0.378321,42936.805095,3.391635,0.097791,0.024081,0.014822,5287.096609,6.296430e+04,0.009584
std,32557.861859,32624.286368,32489.537836,32558.025841,9628.280507,2.517477e+08,2.731712e+05,4789.443879,1.248870e+08,2.447237,...,0.183893,0.484969,32185.565297,3.278856,0.297032,0.153301,0.120839,18195.690344,2.526503e+05,0.097427
min,3166.110000,3174.780000,3156.260000,3167.070000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,2018.000000,...,0.000000,0.000000,3.000000,1.000000,0.000000,0.000000,0.000000,138.987163,3.489000e+02,0.000000
25%,9694.630000,9726.740000,9660.740000,9694.650000,220.138490,4.994906e+06,6.178000e+03,108.218680,2.384668e+06,2020.000000,...,0.000000,0.000000,14161.000000,1.000000,0.000000,0.000000,0.000000,1032.969019,6.973550e+03,0.000000
50%,29325.900000,29379.830000,29276.880000,29326.690000,469.253358,1.402873e+07,1.555700e+04,232.868270,6.812510e+06,2022.000000,...,0.000000,0.000000,36057.000000,2.000000,0.000000,0.000000,0.000000,1888.972847,1.701705e+04,0.000000
75%,62212.790000,62405.300000,62037.800000,62213.840000,1261.560966,3.799360e+07,4.372500e+04,629.555299,1.873743e+07,2024.000000,...,0.000000,1.000000,71219.000000,4.000000,0.000000,0.000000,0.000000,4468.379683,4.658350e+04,0.000000
max,126011.180000,126199.630000,125648.010000,126011.180000,760705.362783,1.746531e+10,1.522359e+07,374775.574085,8.783916e+09,2026.000000,...,1.000000,1.000000,105317.000000,34.000000,1.000000,1.000000,1.000000,437774.382210,9.914953e+06,1.000000


In [49]:
cols_to_keep = [

# Time
'Open time','timeframe',
'year','quarter','month','day',

# Price data
'Open','High','Low','Close',
'price_change','return_pct','volatility',

# Volume & market activity
'Volume','capital_flow','trade_activity',
'buy_volume','sell_volume',
'buy_ratio','sell_ratio','order_flow',

# Trend KPIs
'trend_direction','trend_length','trend_strength',

 'avg_volume_20',
 'avg_volatility_20',

# Market behavior signals
'volume_spike',
'high_volatility',
'high_volatility_length',
'compression',
'overheating',
'whale_activity',

# Breakouts
'breakout_up',
'breakout_down',

# Support & resistance
'support',
'resistance'
]


In [ ]:
# Review dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391385 entries, 0 to 391384
Data columns (total 44 columns):
 #   Column                        Non-Null Count   Dtype              
---  ------                        --------------   -----              
 0   Open time                     391385 non-null  datetime64[ns, UTC]
 1   Open                          391385 non-null  float64            
 2   High                          391385 non-null  float64            
 3   Low                           391385 non-null  float64            
 4   Close                         391385 non-null  float64            
 5   Volume                        391385 non-null  float64            
 6   Close time                    391385 non-null  datetime64[ns, UTC]
 7   capital_flow                  391385 non-null  float64            
 8   trade_activity                391385 non-null  int64              
 9   buy_volume                    391385 non-null  float64            
 10  Taker buy quote asse

In [52]:
df_final = df[cols_to_keep]
df_final.to_csv(r"C:\Users\admin\Desktop\silver_bitcoin.csv", index=False)

df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391385 entries, 0 to 391384
Data columns (total 36 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   Open time               391385 non-null  datetime64[ns, UTC]
 1   timeframe               391385 non-null  object             
 2   year                    391385 non-null  int64              
 3   quarter                 391385 non-null  int64              
 4   month                   391385 non-null  int64              
 5   day                     391385 non-null  int64              
 6   Open                    391385 non-null  float64            
 7   High                    391385 non-null  float64            
 8   Low                     391385 non-null  float64            
 9   Close                   391385 non-null  float64            
 10  price_change            391385 non-null  float64            
 11  return_pct              39